# Cryptographic Hashing

A cryptographic hash function takes input data and produces a fixed-size "fingerprint." The same input always produces the same hash, but even a tiny change in the input produces a completely different result. Hashes are one-way — you cannot recover the original data from a hash.

eXist-db 7.0 provides two hash functions: [`fn:hash()`]({docs}/functions/fn/hash) from the XQuery 4.0 standard, and [`crypto:hash()`]({docs}/functions/crypto/hash) from the EXPath Cryptographic Module. This chapter starts with `fn:hash()` and then shows where `crypto:hash()` adds capabilities you might need.

## fn:hash — The Standard Approach

[`fn:hash()`]({docs}/functions/fn/hash) is the W3C XQuery 4.0 standard function. It returns `xs:hexBinary`, which displays as an uppercase hex string:

In [ ]:
fn:hash("Hello, World!", "SHA-256")

With just one argument, it defaults to MD5:

In [ ]:
fn:hash("Hello, World!")

## Comparing Algorithms

[`fn:hash()`]({docs}/functions/fn/hash) supports seven algorithms, from fast checksums to modern cryptographic hashes:

In [ ]:
xquery version "4.0";

for $alg in ("MD5", "SHA-1", "SHA-256", "SHA-384", "SHA-512", "CRC-32", "BLAKE3")
return
    <hash algorithm="{$alg}"
          bits="{string-length(string(fn:hash('test', $alg))) * 4}"
          value="{fn:hash('test', $alg)}"/>

SHA-256 is the most widely used. MD5 and SHA-1 are considered weak for security purposes but are still useful for checksums. CRC-32 is a non-cryptographic checksum — fast, but not suitable for security. BLAKE3 is a modern algorithm that is both fast and secure.

## Verifying Data Integrity

Hashes are commonly used to verify that data hasn't been corrupted or tampered with:

In [ ]:
xquery version "4.0";

let $original := "The quick brown fox jumps over the lazy dog"
let $expected-hash := fn:hash($original, "SHA-256")

(: Later, verify the data is unchanged :)
let $received := "The quick brown fox jumps over the lazy dog"
let $actual-hash := fn:hash($received, "SHA-256")

return
    <integrity-check>
        <expected>{$expected-hash}</expected>
        <actual>{$actual-hash}</actual>
        <match>{$expected-hash = $actual-hash}</match>
    </integrity-check>

## Hashing Binary Data

[`fn:hash()`]({docs}/functions/fn/hash) accepts `xs:base64Binary` and `xs:hexBinary` input, hashing the raw bytes rather than a string representation:

In [ ]:
xquery version "4.0";

let $binary := xs:base64Binary("SGVsbG8=")  (: "Hello" as base64 :)
return fn:hash($binary, "SHA-256")

## crypto:hash — When You Need More

[`crypto:hash()`]({docs}/functions/crypto/hash) from the EXPath Cryptographic Module offers capabilities beyond `fn:hash()`.

### Base64 Output

The most common reason to reach for `crypto:hash()` is when you need base64-encoded output directly — for example, when constructing OAuth signatures or HTTP authentication headers. `fn:hash()` returns `xs:hexBinary`; getting base64 from it requires a conversion step. `crypto:hash()` returns base64 by default:

In [ ]:
import module namespace crypto = "http://expath.org/ns/crypto";

(: Base64 — the default :)
crypto:hash("Hello, World!", "SHA-256"),

(: Hex — pass "hex" as the third argument :)
crypto:hash("Hello, World!", "SHA-256", "hex")

### Hashing XML Nodes

`crypto:hash()` accepts XML nodes directly, hashing their string value:

In [ ]:
import module namespace crypto = "http://expath.org/ns/crypto";

let $doc := <order id="123"><total>99.99</total></order>
return crypto:hash($doc, "SHA-256", "hex")

### Cross-Engine Portability

`crypto:hash()` conforms to the [EXPath Cryptographic Module](http://expath.org/spec/crypto) specification. BaseX also implements it, so code using `crypto:hash()` is portable across engines — unlike `fn:hash()`, which requires XQuery 4.0 support.

## Side-by-Side Comparison

Both functions produce the same digest for the same input. The difference is in the return type:

In [ ]:
xquery version "4.0";
import module namespace crypto = "http://expath.org/ns/crypto";

let $input := "test"
let $fn-hex := string(fn:hash($input, "SHA-256"))
let $crypto-hex := crypto:hash($input, "SHA-256", "hex")
return
    <comparison>
        <fn-hash type="xs:hexBinary">{$fn-hex}</fn-hash>
        <crypto-hash type="xs:string">{$crypto-hex}</crypto-hash>
        <same-digest>{upper-case($crypto-hex) = $fn-hex}</same-digest>
    </comparison>

## Which Should You Use?

| | [`fn:hash()`]({docs}/functions/fn/hash) | [`crypto:hash()`]({docs}/functions/crypto/hash) |
|---|---|---|
| **Standard** | W3C XQuery 4.0 | EXPath Cryptographic Module |
| **Returns** | `xs:hexBinary` | `xs:string` (base64 or hex) |
| **Algorithms** | MD5, SHA-1, SHA-256, SHA-384, SHA-512, CRC-32, BLAKE3 | MD5, SHA-1, SHA-256, SHA-384, SHA-512 |
| **Base64 output** | Requires conversion | Built-in (default) |
| **Node input** | No | Yes |
| **Portability** | XQuery 4.0 engines | eXist-db, BaseX |

**For new code**, prefer `fn:hash()` — it's the standard. **Reach for `crypto:hash()`** when you need base64 output, node input, or cross-engine portability.